# Test de Sistemas Cuánticos

Se tiene como primeros objetivos:
- Tomar un potencial y construir la ecuación de Schödinger.
- Resolver la ecuación con distintas estrategias dependiendo del tipo de potencial.
- Aplicar las condiciones de frontera a la solución general.
- Cuantizar la energía y calcular estos valores permitidos, así como la energía del estado fundamental
- Obtener un estado de energía $n$ y regresar una instancia de la clase `WaveFunction` caraterística $\psi_{n}(x)$

Para comenzar, resolveremos sistemas cuánticos sencillos que impliquen potenciales sencillos como el del pozo de potencial infinito, pero sin resolver dicho potencial a trozos, si no tomar solo la parte que nos interesa.

In [ ]:
import sympy as sp

sp.init_printing()

x = sp.symbols("x", real=True)
E = sp.symbols("E", positive=True)
C1, C2, L, m, hbar = sp.symbols("C_1 C_2 L m hbar", positive=True)
n = sp.symbols("n", integer=True, nonnegative=True)

psi = sp.Function("psi")

## Pozo de potencial infinito

Supondremos un potencial $V(x) = 0$ en un dominio de $[0, L]$

In [ ]:
V = sp.S.Zero
T = - hbar**2 / (2*m) * sp.diff(psi(x), x, 2)

H = T + V
ecuación = sp.Eq(E * psi(x), H)
ecuación

Este es un potencial que puede ser resuelto facilmente con `sp.dsolve()`, y miremos primero como queda la ecuación general y luego al aplicar las condiciones de frontera como argumento con `ics=`, las cuales son:

$$
\psi (0) = 0, \psi (L) = 0
$$

In [ ]:
solución_general = sp.dsolve(ecuación, psi(x))
display(solución_general)

solución_general_ics = sp.dsolve(ecuación, psi(x), ics={psi(0):0, psi(x).subs(x, L):0})
display(solución_general_ics)

Como podemos ver, al aplicar las condiciones de frontera directamente en `sp.dsolve()` el método toma directamente la solución trivial, por lo tanto, tendremos que aplicar estas condiciones de frontera de manera manual.

Para ello, tomaremos el lado derecho de la solución general y extraeremos los símbolos, particularmente las constantes, de esta y les asignaremos sus correspondientes suposiciones. Pero primero simplifiquemos algunas constantes mediante k.

In [ ]:
expr = solución_general.rhs

# Simplificamos
k = sp.Symbol('k', positive=True)
k_expr = sp.sqrt(2*m*E)/hbar
expr = expr.subs(k_expr, k)

constantes_originales = expr.free_symbols - {x, E, hbar, m}

In [ ]:
# Dado que devuelve un conjunto, los transformamos a lista
constantes_originales = list(sorted(constantes_originales, key=str)) # Ordenamos para asegurarnos que esten ordenadas

# Añadimos las suposiciones a las constantes con el mismo nombre de las extraidas
C1 = sp.Symbol(constantes_originales[0].name, real=True, nonzero=True)
C2 = sp.Symbol(constantes_originales[1].name, real=True)
constantes = sorted([C1, C2], key=str)

expr = expr.subs({
    constantes_originales[0]: constantes[0],
    constantes_originales[1]: constantes[1]
})

Y volvemos a tener la misma solución general pero con suposiciones asignadas a las constantes, de modo que no debería devolvernos la solución trivial. 

Tratemos de obtener nuevamente la solución aplicando las condiciones de forntera.

In [ ]:
# Ecuaciones a resolver
condiciones = [
    sp.Eq(expr.subs(x, 0), 0, evaluate=False),
    sp.Eq(expr.subs(x, L), 0, evaluate=False)
]

condiciones

In [ ]:
constantes_sol = sp.solve(
    condiciones,
    constantes,
    dict=True
)

constantes_sol

Como podemos ver, nos devuelve una lista vacía. Esto se debe a que SymPy aún trata de devolvernos la solución trivial pero ahora se contradice con las suposiciones que hicimos a las constantes, particularmente $C_1 \neq 0$.

Entonces, trataremos de aplicar dichas condiciones de frontera mediante un sistema de ecuaciones y calculando los eigenvalores y eigenvectores.

En primer lugar, para crear la matriz de funciones sin las constantes, aplicaremos la derivada a la ecuación sobre estas de modo que obtengamos solo los coeficientes de estas, es decir, para que estas se eliminen y nos quede una matriz sin las constantes.

In [ ]:
A = sp.Matrix([
    [sp.diff(condicion.lhs, constante) for constante in constantes] for condicion in condiciones
])
# Observemos que se tomó el lado izquierdo pues condiciones es una lista de ecuaciones

A

Creamos la matriz de constantes y ya tendríamos nuestro sistema de ecuaciones.

In [ ]:
C = sp.Matrix([
    C1,
    C2
])

display(sp.Eq(A * C, 0, evaluate=False))

Ahora sí, si el determinante es distinto de cero entonces podemos encontrar soluciones no triviales. Asi que primero encontremos los valores de k permitidos y de ahí sustituimos E para encontrar los valores de energía permitidos, recordando que:
$$
k^2 = \frac{2 m E}{\hbar ^2}, E = \frac{\hbar ^2 k ^2}{2m}
$$

In [ ]:
n = sp.symbols('n', integer=True, positive=True)
det_A = sp.simplify(A.det())

if sp.simplify(det_A) != 0:
    sol_k = sp.solveset(
    det_A,
    k,
    domain=sp.S.Reals
)

sp.simplify(sol_k)

Obtenemos un conjunto como solución, sin embargo, al tratar de usarlo para encontrar los valores de energía permitidos será muy complicado trabajar con este, por lo que mejor regresemos un poco y obtengamos la ecuación característica y trabajemos con ella.

In [ ]:
ecuacion_caracteristica = sp.Eq(det_A, 0).simplify()
display("Ecuación caraterística: ", ecuacion_caracteristica)

# Regresamos a la variable E
ecuacion_E = ecuacion_caracteristica.subs(k, k_expr)

# Resolvemos la ecuación para los valores de E
sol_E = sp.solveset(
    ecuacion_E,
    E,
    domain=sp.S.Reals
)

display("Niveles de energía permitidos: ", sp.simplify(sol_E))
print(type(sol_E))

Hasta el momento logramos obtener un conjunto con los niveles de energía permitidos, sin embargo, falta implementar alguna manera de mejorar este resultado para poderlo manejar facilmente a la hora de tanto calcular las funciones de onda como graficar estos niveles de energía y estados.

Una primera opción podría ser obtener el argumento del seno de la ecuación característica y, dado que el seno debe ser cero, entonces buscamos las soluciones tal que su argumento sea $n * \pi$

In [ ]:
# Resolvemos la ecuación para los valores de E
sol_E = sp.solveset(
    sp.Eq(ecuacion_E.lhs.args[0], n * sp.pi),
    E,
    domain=sp.S.Reals
)

formula_sol_E = list(sol_E)

display(sp.Eq(sp.Symbol("E_n"), sp.simplify(formula_sol_E[0]), evaluate=False))

## Futruas implementaciones

A continuación algunos puntos los cuales serían buenos agregar, para ello se deberá investigar y justificar su uso.

- ``picewise()`` para funciones a trozos
- argumento ``hint=`` en ``dsolve()`` para ayudar a encontrar la solucion
- Coeficientes de transmision y reflexion para sistemas de dispersión
- Densidad de corriente de probabilidad $J(x) = \frac{\hbar}{m} Im(\psi^* \frac{d\psi}{dx})$
donde se usará ``sp.im()`` para extraer la parte imaginaria